# 0.5 State Limit Derivation

This notebook derives candidate state-growth targets and limits from configurable annual growth profiles.

It does not use historical blocks, does not run passive replay, and does not implement Mechanism A or Mechanism B.

In [1]:
from dataclasses import asdict, replace
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from state_limits import (
    BRIEF_100GIB_CPSB1174,
    CURRENT_EIP8037_120GIB_CPSB1530,
    derive_state_limit,
)

## Formula

The state limit module converts an annual state-growth budget into a per-block state gas target and hard limit:

```text
target_bytes_per_block = annual_growth_gib * 2**30 / blocks_per_year
state_gas_target       = int(target_bytes_per_block * cpsb)
state_gas_limit        = state_gas_target * limit_target_ratio
```

The default block count is `2,628,000` blocks per year under 12s slot time.

In [2]:
profiles = [
    BRIEF_100GIB_CPSB1174,
    CURRENT_EIP8037_120GIB_CPSB1530,
]

state_limits = pd.DataFrame([asdict(derive_state_limit(profile)) for profile in profiles])
state_limits

,profile_name,annual_growth_gib,cpsb,blocks_per_year,limit_target_ratio,target_bytes_per_block,state_gas_target,state_gas_limit
0,brief_100gib_cpsb1174,100.0,1174,2628000,2,40857.755860,47967005,95934010
1,current_eip8037_120gib_cpsb1530,120.0,1530,2628000,2,49029.307032,75014839,150029678


## Limit Ratio Sensitivity

Changing `limit_target_ratio` changes the hard limit but leaves the target unchanged.

In [3]:
base_profile = BRIEF_100GIB_CPSB1174
profiles_by_ratio = [
    replace(base_profile, limit_target_ratio=ratio)
    for ratio in [2, 3, 4]
]

pd.DataFrame([asdict(derive_state_limit(profile)) for profile in profiles_by_ratio])

,profile_name,annual_growth_gib,cpsb,blocks_per_year,limit_target_ratio,target_bytes_per_block,state_gas_target,state_gas_limit
0,brief_100gib_cpsb1174,100.0,1174,2628000,2,40857.75586,47967005,95934010
1,brief_100gib_cpsb1174,100.0,1174,2628000,3,40857.75586,47967005,143901015
2,brief_100gib_cpsb1174,100.0,1174,2628000,4,40857.75586,47967005,191868020
